In [ ]:
# Module 11 — Code Along: Tools as Python functions + the agent loop (support-ticket triage)
# SHARED domain + FakeLLM block — identical across module-10/11/12. Paste verbatim.
from __future__ import annotations
import json
from types import SimpleNamespace
from typing import Optional, Any, Callable
from dataclasses import dataclass, field
from pydantic import BaseModel, Field, ValidationError

@dataclass
class SupportTicket:
    id: int
    subject: str
    category: str          # "Billing" | "Technical" | "Account"
    priority: str          # "low" | "normal" | "high" | "urgent"
    is_open: bool = True
    tags: list[str] = field(default_factory=list)

SAMPLE_TICKETS = [
    SupportTicket(1, "Can't log in after password reset", "Account", "high"),
    SupportTicket(2, "Invoice double-charged this month",  "Billing", "urgent"),
    SupportTicket(3, "How do I export my data?",            "Technical", "low"),
    SupportTicket(4, "App crashes on upload",               "Technical", "high"),
    SupportTicket(5, "Refund not received",                 "Billing", "normal", is_open=False),
]

# --- the dependency-injection seam: a fake stand-in for openai.OpenAI ---
def _msg(content=None, tool_calls=None):
    return SimpleNamespace(content=content, tool_calls=tool_calls)
def _resp(message):
    return SimpleNamespace(choices=[SimpleNamespace(message=message)])
def _tool_call(call_id, name, **args):
    return SimpleNamespace(id=call_id,
        function=SimpleNamespace(name=name, arguments=json.dumps(args)))

class FakeLLM:
    """Scripts chat.completions.create — same shape as openai.OpenAI, no API key.
    Exactly the Day-3 pattern: inject a fake at the seam instead of the real client."""
    def __init__(self, responses):
        self._responses = list(responses)
        self.chat = SimpleNamespace(
            completions=SimpleNamespace(create=self._create))
    def _create(self, **kwargs):
        return self._responses.pop(0)

print(f"{len(SAMPLE_TICKETS)} sample tickets loaded; FakeLLM ready (no API key needed).")

## The big idea

**A tool is just a Python function the LLM is allowed to call.** No framework, no magic.

- The LLM never runs your code. It *names* a function + JSON arguments; **you** run it and hand back the result.
- The "agent" is a `while` loop in plain Python around the model — the model plans, your code acts.
- Everything below is ~80 lines of Python you can read top to bottom. The cleverness is in the loop, not a library.

In [ ]:
# 2 · The tool plumbing — a spec (name + description + JSON-schema params + the fn)
#       and a registry that holds them and emits the schema the LLM sees.
@dataclass
class ToolSpec:
    name: str
    description: str
    parameters: dict
    fn: Callable[..., Any]
    def to_openai_schema(self) -> dict:
        return {"type": "function", "function": {
            "name": self.name, "description": self.description,
            "parameters": self.parameters}}

class ToolRegistry:
    def __init__(self):
        self._tools: dict[str, ToolSpec] = {}
    def tool(self, *, name, description, parameters):
        def deco(fn):
            self._tools[name] = ToolSpec(name, description, parameters, fn)
            return fn
        return deco
    def get(self, name) -> ToolSpec:
        if name not in self._tools:
            raise KeyError(f"tool {name!r} not registered")
        return self._tools[name]
    def openai_schemas(self):
        return [t.to_openai_schema() for t in self._tools.values()]

registry = ToolRegistry()
print("empty registry ready:", registry.openai_schemas())

## A tool is the M5 decorator, reimagined

`@registry.tool(name=..., description=..., parameters=...)` is exactly the decorator pattern from Day 1/2 (M5):

- it **stamps metadata** onto a plain function (the name + the JSON schema the LLM reads), and
- it **registers** the function so the agent can find and call it later.

Same `@decorator` you already met — the function is unchanged and still directly callable; the decorator just records it.

In [ ]:
# 4 · Register ONE standalone tool, then prove it's nothing but a Python function.
@registry.tool(
    name="count_by_priority",
    description="Count tickets grouped by priority.",
    parameters={"type": "object", "properties": {}, "required": []})
def count_by_priority():
    out: dict[str, int] = {}
    for t in SAMPLE_TICKETS:
        out[t.priority] = out.get(t.priority, 0) + 1
    return out

# (a) it's still just a function — call it directly, no LLM involved:
print("direct call:", registry.get("count_by_priority").fn())

# (b) this is the JSON the LLM actually sees for that tool:
print("\nschema the LLM sees:")
print(json.dumps(registry.openai_schemas(), indent=2))

## The agent loop

**plan → act → observe → repeat.**

1. **plan** — send messages + tool schemas to the model; it replies with either a final answer or `tool_calls`.
2. **act** — for each requested call, run the real Python function.
3. **observe** — append each result back as a `tool` message so the model can read it.
4. **repeat** until the model answers with plain text (no more tool calls).

`max_steps` is the safety net: a confused model could loop forever, so we cap the turns and raise instead.

In [ ]:
# 6 · Result types + two tiny helpers. The helpers re-encode the model's reply
#     into the shape the NEXT API call expects -- pulling them out here keeps the
#     ask() loop in the next cell readable: plan -> act -> observe.
class AgentError(RuntimeError):
    pass

@dataclass
class ToolCallRecord:
    tool: str
    arguments: dict
    result: Any

@dataclass
class AgentResult:
    answer: str
    tool_calls: list
    steps: int

SYSTEM_PROMPT = ("You are a support-desk assistant. Use the tools to answer "
                 "questions about support tickets. Prefer one accurate tool call.")

def _assistant_turn(msg) -> dict:
    """Re-encode the model's tool-call turn so the next API call accepts it."""
    return {"role": "assistant", "content": msg.content,
            "tool_calls": [{"id": tc.id, "type": "function",
                "function": {"name": tc.function.name,
                             "arguments": tc.function.arguments}}
                for tc in msg.tool_calls]}

def _tool_reply(call, result) -> dict:
    """One tool message per call, tagged with its tool_call_id — the chaining
    slide 11.5 warns about: every reply points back at the call that asked."""
    return {"role": "tool", "tool_call_id": call.id,
            "name": call.function.name,
            "content": json.dumps(result, default=str)}

print("result types + message helpers ready")

In [ ]:
# 6 (cont.) · The canonical TicketAgent -- identical in module-12. Read ask() closely:
#             plan -> act -> observe -> repeat, capped by max_steps.
class TicketAgent:
    def __init__(self, store, llm_client, *, model="gpt-4o-mini", max_steps=5):
        self.store = store
        self.llm = llm_client
        self.model = model
        self.max_steps = max_steps
        self.registry = self._build_registry()

    def _build_registry(self) -> ToolRegistry:
        reg = ToolRegistry()

        @reg.tool(name="list_open_tickets",
                  description="List all currently open tickets.",
                  parameters={"type": "object", "properties": {}, "required": []})
        def list_open_tickets():
            return [t.__dict__ for t in self.store if t.is_open]

        @reg.tool(name="search_tickets",
                  description="Find tickets whose subject contains a substring (case-insensitive).",
                  parameters={"type": "object",
                              "properties": {"term": {"type": "string"}},
                              "required": ["term"]})
        def search_tickets(term):
            return [t.__dict__ for t in self.store if term.lower() in t.subject.lower()]

        @reg.tool(name="count_by_priority",
                  description="Count tickets grouped by priority.",
                  parameters={"type": "object", "properties": {}, "required": []})
        def count_by_priority():
            out: dict[str, int] = {}
            for t in self.store:
                out[t.priority] = out.get(t.priority, 0) + 1
            return out

        return reg

    def ask(self, user_prompt: str) -> AgentResult:
        messages = [{"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": user_prompt}]
        log = []
        for step in range(1, self.max_steps + 1):
            # PLAN -- model sees the conversation + tool schemas, then replies
            resp = self.llm.chat.completions.create(
                model=self.model, messages=messages,
                tools=self.registry.openai_schemas())
            msg = resp.choices[0].message

            if not (msg.tool_calls or []):              # no tools asked -> final answer
                return AgentResult((msg.content or "").strip(), log, step)

            messages.append(_assistant_turn(msg))       # record the model's request
            # ACT + OBSERVE -- run each requested tool, feed its result back
            for call in msg.tool_calls:
                result = self._invoke(call.function.name, call.function.arguments)
                log.append(ToolCallRecord(call.function.name,
                    json.loads(call.function.arguments or "{}"), result))
                messages.append(_tool_reply(call, result))

        raise AgentError(f"did not converge in {self.max_steps} steps")

    def _invoke(self, name, args_json):
        try:
            spec = self.registry.get(name)
        except KeyError:                       # unknown tool -> error observation, not a crash
            return {"error": f"unknown tool: {name!r}"}
        return spec.fn(**(json.loads(args_json or "{}")))

print("TicketAgent defined -- tools:", list(TicketAgent(SAMPLE_TICKETS, FakeLLM([])).registry._tools))

## Watch one turn

The assistant message carries `tool_calls`; then we append **one `{"role":"tool", "tool_call_id": ...}` message per result**.

That id-for-id pairing is the chaining the deck's **slide 11.5** warns about: every tool reply must point back at the call that asked for it, or the next API call rejects the conversation.

In [ ]:
# 8 · THE demo the deck points to (module-11 cell 8): script two model turns, run ask().
#     Turn 1 → model asks for count_by_priority.  Turn 2 → model gives the final answer.
scripted = FakeLLM([
    _resp(_msg(tool_calls=[_tool_call("c1", "count_by_priority")])),
    _resp(_msg(content="You have 2 high, 1 urgent, 1 normal, 1 low ticket.")),
])

agent = TicketAgent(SAMPLE_TICKETS, scripted)
r = agent.ask("break tickets down by priority")

print("answer:", r.answer)
print("steps :", r.steps)
print("calls :", [(c.tool, c.arguments) for c in r.tool_calls])
print("observed result:", r.tool_calls[0].result)

## Why `tool_call_id` chaining is non-negotiable

When the assistant requests N tools, the API expects **exactly N tool messages back, each tagged with its `tool_call_id`** before the next completion.

- The id is how the model matches *this answer* to *that request* — drop it (or mismatch it) and the next `create()` call **400s**.
- Our loop guarantees the pairing: we build the assistant `tool_calls` list, then emit one `tool` message per `call.id` in the same order.
- This is bookkeeping you own in plain Python — not something the library does for you.

In [ ]:
# 10 · A second demo — a tool that takes an argument (search_tickets term="log in").
#      Shows the loop walking plan → act → observe with real arguments flowing through.
scripted2 = FakeLLM([
    _resp(_msg(tool_calls=[_tool_call("c1", "search_tickets", term="log in")])),
    _resp(_msg(content="One open ticket matches: #1 'Can't log in after password reset'.")),
])

agent2 = TicketAgent(SAMPLE_TICKETS, scripted2)
r2 = agent2.ask("any tickets about login problems?")

rec = r2.tool_calls[0]
print("plan   → tool:", rec.tool, "| arguments:", rec.arguments)
print("observe→ result:")
print(json.dumps(rec.result, indent=2, default=str))
print("\nfinal answer:", r2.answer, "| steps:", r2.steps)

## `max_steps` and what comes next

- `max_steps` guards a confused model: if it never stops calling tools, the loop raises `AgentError` instead of running forever.
- Bigger ideas later — **memory, RAG, multi-tool plans** — are *more Python around the LLM*, not more magic. Same loop, richer tools and context.

**→ Lab 11** builds this exact loop on the `Product` / `CatalogAgent` spine: same `ToolRegistry`, same plan→act→observe, your own catalog tools.

## Appendix — the same agent, with a framework

Everything above is ~80 lines of plain Python: the registry, the `tool_call_id` chaining, the loop, the `max_steps` guard. **Frameworks collapse exactly that orchestration** — they derive each tool's JSON schema from your function's *type hints + docstring*, then run the loop for you. Your tools stay plain Python over the data; only the plumbing disappears.

Two below, on the **same SupportTicket domain**. They call a *real* model (the `FakeLLM` trick doesn't apply), so they're **shown, not run**: each cell imports cleanly, and only calls the model if `OPENAI_API_KEY` is set. **Paste a key in the config cell just below to run them live.** Run order: cell 0 (the shared block) first, so `SAMPLE_TICKETS` exists.

In [ ]:
# ── OPTIONAL: run the appendix live ────────────────────────────────
# Paste your OpenAI API key between the quotes to actually call the model in the
# two cells below. Leave it "" to keep them in shown-not-run mode (no network).
# ⚠️  Don't commit or share the notebook with your key pasted here.
OPENAI_API_KEY = ""          # e.g. "sk-proj-..."

import os
if OPENAI_API_KEY:
    os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
    print("key set ✓  — the Pydantic AI / OpenAI Agents cells will call the model")
else:
    print('no key — paste one above to run the appendix, or leave "" for shown-not-run')

In [ ]:
# APPENDIX A · Pydantic AI -- note RunContext.deps IS the Day-3 dependency-injection
# seam: we inject the ticket store as `deps`, exactly like we injected the APIClient.
import os
try:
    from pydantic_ai import Agent, RunContext

    tickets_agent = Agent(
        "openai:gpt-4o-mini",
        deps_type=list,                                  # the injected ticket store
        system_prompt="Answer questions about support tickets using the tools.",
    )

    @tickets_agent.tool
    def search_tickets(ctx: RunContext[list], term: str) -> list[dict]:
        """Find tickets whose subject contains `term` (case-insensitive)."""
        return [t.__dict__ for t in ctx.deps if term.lower() in t.subject.lower()]

    @tickets_agent.tool
    def count_by_priority(ctx: RunContext[list]) -> dict:
        """Count tickets grouped by priority."""
        out = {}
        for t in ctx.deps:
            out[t.priority] = out.get(t.priority, 0) + 1
        return out

    if os.getenv("OPENAI_API_KEY"):
        r = tickets_agent.run_sync("break the tickets down by priority", deps=SAMPLE_TICKETS)
        print(r.output)
    else:
        print("pydantic-ai imported OK. Paste your key in the config cell above to run it.")
except ImportError:
    print("pip install pydantic-ai   # then set OPENAI_API_KEY to run this cell")

In [ ]:
# APPENDIX B · OpenAI Agents SDK -- @function_tool derives the JSON schema from the
# signature + docstring, so the parameters={...} dict you wrote by hand is GONE.
import os
try:
    from agents import Agent, Runner, function_tool

    @function_tool
    def list_open_tickets() -> list[dict]:
        """List all currently open tickets."""
        return [t.__dict__ for t in SAMPLE_TICKETS if t.is_open]

    @function_tool
    def search_tickets(term: str) -> list[dict]:
        """Find tickets whose subject contains `term` (case-insensitive)."""
        return [t.__dict__ for t in SAMPLE_TICKETS if term.lower() in t.subject.lower()]

    @function_tool
    def count_by_priority() -> dict:
        """Count tickets grouped by priority."""
        out = {}
        for t in SAMPLE_TICKETS:
            out[t.priority] = out.get(t.priority, 0) + 1
        return out

    support_agent = Agent(
        name="Support desk assistant",
        instructions="Answer questions about support tickets using the tools.",
        tools=[list_open_tickets, search_tickets, count_by_priority],
        model="gpt-4o-mini",
    )

    if os.getenv("OPENAI_API_KEY"):
        result = Runner.run_sync(support_agent, "how many tickets by priority?")
        print(result.final_output)
    else:
        print("openai-agents imported OK. Paste your key in the config cell above to run it.")
except ImportError:
    print("pip install openai-agents   # then set OPENAI_API_KEY to run this cell")

## What disappeared — and why we still built it by hand

| Lab 11, by hand | Framework |
|---|---|
| `ToolSpec` + `ToolRegistry` + `@registry.tool(parameters={...})` | `@function_tool` / `@agent.tool` — schema auto-derived from signature + docstring |
| `ask()` loop, plan → act → observe | `Runner.run_sync(...)` / `agent.run_sync(...)` |
| append assistant msg **with** `tool_calls`; one `tool` msg per `tool_call_id` | hidden — the #1 breakage, handled for you |
| `max_steps` guard → `AgentError` | built-in turn limit |
| `_invoke_tool` lookup + `{"error": …}` on unknown/raise | built-in |
| **~80 lines** | **~15 lines** |

**The framework removes the typing, not the understanding.** When a real agent breaks — a `tool_call_id` 400, a runaway loop, a tool that silently returns `{}` — whoever hand-rolled the loop debugs it in minutes. You built it once so the framework is never magic. The tools themselves — plain Python over your data — are identical either way.